In [1]:
#reading file from aws minio S3
from pyspark import SparkConf
from pyspark.sql import SparkSession
import os



## DEFINE SENSITIVE VARIABLES
NESSIE_URI = os.environ.get("NESSIE_URI") ## Nessie Server URI
WAREHOUSE = os.environ.get("WAREHOUSE") ## BUCKET TO WRITE DATA TOO
AWS_ACCESS_KEY_ID = os.environ.get("AWS_ACCESS_KEY_ID") ## AWS CREDENTIALS
AWS_SECRET_ACCESS_KEY = os.environ.get("AWS_SECRET_ACCESS_KEY") ## AWS CREDENTIALS
AWS_S3_ENDPOINT= os.environ.get("AWS_S3_ENDPOINT") ## MINIO ENDPOINT
AWS_REGION= os.environ.get("AWS_REGION") ## MINIO ENDPOINT

print(AWS_S3_ENDPOINT)
print(NESSIE_URI)
print(WAREHOUSE)
print(AWS_ACCESS_KEY_ID)
print(AWS_SECRET_ACCESS_KEY)
print(AWS_REGION)

http://minio:9000
http://nessie:19120/api/v1
s3a://warehouse-ny-bike
admin
your_password
us-east-1


In [2]:
print(os.environ.get("MINIO_USER"))

admin


In [5]:
conf = (
    SparkConf()
        .setAppName('Minio_S3_reading_file')
        .set('spark.jars.packages','org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262')
        .set("fs.s3a.endpoint", "http://minio:9000")
        .set('spark.hadoop.fs.s3a.access.key', os.environ.get("AWS_ACCESS_KEY_ID"))
        .set('spark.hadoop.fs.s3a.secret.key', os.environ.get("AWS_SECRET_ACCESS_KEY"))
        .set('spark.hadoop.fs.s3a.path.style.access','true')
        .set('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
)

In [11]:
conf = (
    SparkConf()
        .setAppName('app_name')
        .set('spark.jars.packages','org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0,org.projectnessie.nessie-integrations:nessie-spark-extensions-3.5_2.12:0.102.5,software.amazon.awssdk:bundle:2.20.131,software.amazon.awssdk:url-connection-client:2.20.131,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262') 
        .set('spark.sql.extensions','org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,org.projectnessie.spark.extensions.NessieSparkSessionExtensions')
        .set("fs.s3a.endpoint", "http://minio:9000")
        .set('spark.hadoop.fs.s3a.access.key',AWS_ACCESS_KEY_ID)
        .set('spark.hadoop.fs.s3a.secret.key',AWS_SECRET_ACCESS_KEY)
        # .set('spark.hadoop.fs.s3a.endpoint',AWS_S3_ENDPOINT)
        .set('spark.hadoop.fs.s3a.path.style.access','true')
        .set('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
)

In [3]:

spark = (
    SparkSession.builder
    .appName("ReadFromMinIO")
    .config('spark.jars.packages','org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0,org.projectnessie.nessie-integrations:nessie-spark-extensions-3.5_2.12:0.102.5,software.amazon.awssdk:bundle:2.20.131,software.amazon.awssdk:url-connection-client:2.20.131,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262') 
    .config('spark.sql.extensions','org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,org.projectnessie.spark.extensions.NessieSparkSessionExtensions')    
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "your_password")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)

In [16]:
# spark = SparkSession.builder.config(conf=conf).getOrCreate()

In [8]:
spark

In [4]:
# ✅ Read CSV from MinIO bucket
df = spark.read.csv("s3a://bucket-raw-data/2013-citibike-tripdata/10_October/*", header=True)
list_files=['s3a://bucket-raw-data/raw_data_nybike/2013-citibike-tripdata/10_October/201310-citibike-tripdata_1.csv',
            's3a://bucket-raw-data/raw_data_nybike/2013-citibike-tripdata/10_October/201310-citibike-tripdata_2.csv',
            's3a://bucket-raw-data/raw_data_nybike/2013-citibike-tripdata/11_November/201311-citibike-tripdata_1.csv']
# df = spark.read.csv(list_files, header=True)

26/04/21 08:35:22 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

In [5]:
df.show(2)

+------------+-------------------+-------------------+----------------+--------------------+----------------------+-----------------------+--------------+--------------------+--------------------+---------------------+------+----------+----------+------+
|tripduration|          starttime|           stoptime|start station id|  start station name|start station latitude|start station longitude|end station id|    end station name|end station latitude|end station longitude|bikeid|  usertype|birth year|gender|
+------------+-------------------+-------------------+----------------+--------------------+----------------------+-----------------------+--------------+--------------------+--------------------+---------------------+------+----------+----------+------+
|         326|2013-10-01 00:01:08|2013-10-01 00:06:34|             239|Willoughby St & F...|           40.69196566|            -73.9813018|           366|Clinton Ave & Myr...|           40.693261|           -73.968896| 16052|Subscriber

In [6]:
config={
    'etl_conf':{
        'column_to_add':{'column_name':'dw_period_tag','column_value':102013},
        'cast_to_timestamp':['start_at', 'stop_at'],
        'columns_to_rename':{'starttime':'start_at','stoptime':'stop_at','start station latitude':'start_station_latitude'}
        }
}

In [7]:
import sys
# sys.path.append("/pyspark_etl_pipeline/src")
from transformers_tools.transformers import DataTransformerObject,FactoryDataTransformer,runner_transformer_data
from steps.steps_pipeline import StepsPipelinesEtl


In [8]:
catalog = [
            DataTransformerObject(
                transformer= FactoryDataTransformer.RENAME_COLUMNS,
                config= config['etl_conf']
            ),
            DataTransformerObject(
                transformer= FactoryDataTransformer.ADD_COLUMN_WITH_LITERAL_VALUE,
                config = config['etl_conf']
            ), 
            DataTransformerObject(
                transformer= FactoryDataTransformer.CAST_TO_TIMESTAMP,
                config = config['etl_conf']
            )
        ]

In [9]:
df2 = runner_transformer_data(catalog,df)

Column With value literal initiated
Cast to timestamp: ['start_at', 'stop_at']


In [10]:
df2.show(3)

+------------+-------------------+-------------------+----------------+--------------------+----------------------+-----------------------+--------------+--------------------+--------------------+---------------------+------+----------+----------+------+-------------+
|tripduration|           start_at|            stop_at|start station id|  start station name|start_station_latitude|start station longitude|end station id|    end station name|end station latitude|end station longitude|bikeid|  usertype|birth year|gender|dw_period_tag|
+------------+-------------------+-------------------+----------------+--------------------+----------------------+-----------------------+--------------+--------------------+--------------------+---------------------+------+----------+----------+------+-------------+
|         326|2013-10-01 00:01:08|2013-10-01 00:06:34|             239|Willoughby St & F...|           40.69196566|            -73.9813018|           366|Clinton Ave & Myr...|           40.6932